# Comunidad de Madrid · renta vs coste de vivienda por municipios

Nivel 2 del análisis (el nivel 1 es por barrios de la ciudad, proyecto 01). Unidad: **municipio** de la CAM.

> ⚠️ Por defecto usa **datos SINTÉTICOS** (semilla fija) de los ~30 municipios más poblados (nombres reales, cifras sintéticas). Para datos reales del INE, cambia `SOURCE='real'` (ver README y la guía del proyecto 01).


## 1. Configuración


In [ ]:
import sys, os
SRC = os.path.abspath('../src')
if SRC not in sys.path: sys.path.insert(0, SRC)
import pandas as pd, matplotlib.pyplot as plt
import config
from synthetic import generar_municipios
from transform import add_indicators
SOURCE = 'synthetic'


## 2. Datos


In [ ]:
if SOURCE == 'synthetic':
    base = generar_municipios()
else:
    base = pd.read_csv(config.DATA_PROCESSED / 'cam_municipios.csv')
df = add_indicators(base)
print(f'{len(df)} municipios')
df.sort_values('esfuerzo_alquiler_pct', ascending=False).head()


## 3. ¿Dónde se vive mejor y dónde más apurado?


In [ ]:
cols = ['municipio','renta_hogar','alquiler_eur_m2_mes','esfuerzo_alquiler_pct','esfuerzo_compra_anios','clasificacion_alquiler']
print('MEJOR (menor esfuerzo de alquiler):')
display(df.nsmallest(8,'esfuerzo_alquiler_pct')[cols].reset_index(drop=True))
print('\nMÁS APURADO:')
display(df.nlargest(8,'esfuerzo_alquiler_pct')[cols].reset_index(drop=True))


## 4. Esfuerzo de alquiler por municipio


In [ ]:
d = df.sort_values('esfuerzo_alquiler_pct')
colors = ['#2ca25f' if v<30 else '#fec44f' if v<40 else '#de2d26' for v in d['esfuerzo_alquiler_pct']]
fig, ax = plt.subplots(figsize=(9,9))
ax.barh(d['municipio'], d['esfuerzo_alquiler_pct'], color=colors)
ax.axvline(30, ls='--', c='grey', lw=1); ax.axvline(40, ls='--', c='grey', lw=1)
ax.set_xlabel('% de la renta del hogar en alquiler (80 m2)')
ax.set_title(f'CAM - esfuerzo de alquiler por municipio  [{SOURCE.upper()}]')
plt.tight_layout(); plt.show()


## 5. Renta vs esfuerzo


In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
sc = ax.scatter(df['renta_hogar'], df['esfuerzo_alquiler_pct'], c=df['esfuerzo_compra_anios'], cmap='viridis', s=60)
for _, row in df.iterrows():
    ax.annotate(row['municipio'], (row['renta_hogar'], row['esfuerzo_alquiler_pct']), fontsize=6, alpha=0.7)
ax.set_xlabel('Renta media del hogar (EUR/anio)'); ax.set_ylabel('Esfuerzo de alquiler (%)')
ax.set_title(f'CAM municipios - renta vs esfuerzo  [{SOURCE.upper()}]')
plt.colorbar(sc, label='Anios para comprar (100 m2)'); plt.tight_layout(); plt.show()


## 6. Conclusiones y limitaciones

- El esfuerzo de **alquiler** discrimina mucho entre municipios: el noroeste (Pozuelo, Las Rozas, Boadilla) holgado; el sur (Parla, Fuenlabrada) más apurado.
- El **precio de compra por municipio** tiene buena cobertura oficial (Mº Vivienda), así que el esfuerzo de compra es fiable a este nivel.

**Limitaciones:** datos sintéticos salvo `SOURCE='real'`; demo con ~30 municipios (el INE cubre los 179); fuentes distintas para renta y vivienda.
